In [0]:
## TODO need widgets to pass variables to/from nbs/tasks
# Update paths 

# [**NuInsSeg Dataset**](https://github.com/masih4/NuInsSeg/tree/main?tab=readme-ov-file#nuinsseg--a-fully-annotated-dataset-for-nuclei-instance-segmentation-in-he-stained-histological-images) 

**One of the largest publicly available datasets of segmented nuclei in [H&E-Stained](https://en.wikipedia.org/wiki/H%26E_stain) Histological Images.**

---     
[<img src="https://raw.githubusercontent.com/masih4/NuInsSeg/main/git%20images/prepration.png" width="1000"/>](https://raw.githubusercontent.com/masih4/NuInsSeg/main/git%20images/prepration.png) 

<!-- ![Preparation](https://raw.githubusercontent.com/masih4/NuInsSeg/main/git%20images/prepration.png) -->

![]() 

[<img src="https://raw.githubusercontent.com/masih4/NuInsSeg/main/git%20images/segmentation%20sample.jpg" width="1000"/>](Segmentation Sample](https://raw.githubusercontent.com/masih4/NuInsSeg/main/git%20images/segmentation%20sample.jpg) 
<!-- ![Segmentation Sample](https://raw.githubusercontent.com/masih4/NuInsSeg/main/git%20images/segmentation%20sample.jpg)  -->

---      


**NOTES:**
- MVP_v0 Processing 
- Thumbnails Viz requires classic compute
- Eventually to include [Auto Loader](https://docs.databricks.com/aws/en/machine-learning/reference-solutions/images-etl-inference) to trigger Next Processing Steps wrt `Mask-Contours : CoCo-YOLO Data Formatting`
- Extension to include multi-categorical annotated images e.g. [BraTS](https://www.synapse.org/Synapse:syn51156910/wiki/622351)

### Notebook Dependencies & Paths 

In [0]:
!pip install -q kaggle

%restart_python

In [0]:
# import os
# import copy
# # import random
# # import json
# # import yaml
# # import glob
# # # import cv2
# # import numpy as np
# # import time
# # import matplotlib.pyplot as plt
# # import matplotlib.patches as patches
# # import requests
# # from zipfile import ZipFile
# # import argparse
# # # from PIL import Image
# # # import PIL.Image
# import shutil
# from IPython.display import Image
# # from sklearn.model_selection import train_test_split

# # import torch
# # import torch.utils.data
# # from torch import nn
# # import torchvision

# # from ultralytics import YOLO

# # from torchvision import transforms as T

# # from pycocotools import mask as coco_mask
# # from pycocotools.coco import COCO

In [0]:
CATALOG_NAME = "mmt"
SCHEMA_NAME = "cv"

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}"

# DATA_DIR = f"{VOLUME_PATH}/data"
PROJECTS_DIR = f"{VOLUME_PATH}/projects"

PROJECT_PATH = f"{PROJECTS_DIR}/NuInsSeg"

PROJECT_RAWDATA_DIR = f"{PROJECT_PATH}/raw_data"

ZIPFILE_PATH = f"{PROJECT_RAWDATA_DIR}/nuinsseg.zip"

YOLO_DATA_DIR = f"{PROJECT_PATH}/yolo_dataset" # can update this to change the path to your own data 

# Get the current working directory
nb_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
current_path = f"/Workspace{nb_context}"
# print(f"Current path: {current_path}")
WS_PROJ_DIR = '/'.join(current_path.split('/')[:-1]) 

WORKSPACE_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("user").get()
USER_WORKSPACE_PATH = f"/Users/{WORKSPACE_PATH}"

# project_name = "yolo_MedCellTypes_InstanceSeg"
# experiment_name = USER_WORKSPACE_PATH + "mlflow_experiments/yolo/{project_name}"
# print(f"Setting experiment name to be {experiment_name}")

In [0]:
PROJECT_RAWDATA_DIR

## Download [NuInsSeg dataset from Kaggle.com](https://www.kaggle.com/datasets/ipateam/nuinsseg) to [Unity Catalog Volumes](https://docs.databricks.com/aws/en/volumes/)

We will download the full publicly available dataset hosted on Kaggle.com and expand the zipped files into the respective subdirectories and folders. 

In [0]:
# Create a dropdown widget for run_download
dbutils.widgets.dropdown("run_download", "False", ["True", "False"], "Run Download")

# Get the value of the widget
run_download = dbutils.widgets.get("run_download") == "True" ## because we have already done this there's no need re-run

### Download dataset to UC Volumes path

In [0]:
if run_download:
    # Set up Kaggle API credentials
    import os
    import json
  
    # Set the environment variables
    os.environ['KAGGLE_USERNAME'] = dbutils.secrets.get("mmt", "kaggle_username") ## update to use your own kaggle username
    os.environ['KAGGLE_KEY'] = dbutils.secrets.get("mmt", "kaggle_apikey") ## update to use your own kaggle api key

    # Download the dataset
    !kaggle datasets download -d ipateam/nuinsseg -p {PROJECT_RAWDATA_DIR} #--force

In [0]:
# Dataset URL: https://www.kaggle.com/datasets/ipateam/nuinsseg
# License(s): Attribution 4.0 International (CC BY 4.0)
# Downloading nuinsseg.zip to /Volumes/mmt/cv/projects/NuInsSeg/raw_data
#  99%|██████████████████████████████████████▌| 1.50G/1.52G [00:18<00:00, 104MB/s]
# 100%|██████████████████████████████████████| 1.52G/1.52G [00:19<00:00, 85.5MB/s]

### Inflate zipped folders and files

In [0]:
if run_download:
  ## Unzip the dataset to the Unity Catalog volume using bash 
  dbutils.fs.mkdirs(PROJECT_RAWDATA_DIR)  # Ensure the target directory exists
  !unzip -o {ZIPFILE_PATH} -d {PROJECT_RAWDATA_DIR}  

--------------------------

### [Optional] Check folder paths within subdirectories of interest

In [0]:
directories2use = ['tissue images', 'mask binary without border', 'label masks modify']

# List the contents of directories2use for every subdirectory in PROJECT_RAWDATA_DIR "base_dir"
for subdirectory in dbutils.fs.ls(PROJECT_RAWDATA_DIR):    
    if subdirectory.isDir() and (subdirectory.name != 'yolo_dataset/'): # could be a yolo_datasets base_pathname
        for directory in directories2use:   
            full_path = f"{subdirectory.path}{directory}/"
            print(f"Contents of {full_path}:")  
            # display(dbutils.fs.ls(full_path))

## Visualize Downloaded Image Files

#### Display Images as Thumbnails in a Dataframe

In [0]:
from pyspark.sql import functions as F

# Define the project directory and subdirectories's to use
directories2use = ['tissue images', 'mask binary without border', 'label masks modify']

# List to store all image paths
image_paths = []

# Iterate through the specified subdirectories
for subdirectory in dbutils.fs.ls(PROJECT_RAWDATA_DIR)[2:5]:  # display just one CellType for illustration.  

    if subdirectory.isDir() and (subdirectory.name != 'yolo_dataset/'): # could be a yolo_datasets base_pathname
        for directory in directories2use:   
            full_path = f"{subdirectory.path}{directory}/"
            files = dbutils.fs.ls(full_path)
            for file in files:
                if file.path.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                    image_paths.append(file.path)

# Load all image paths as a DataFrame
image_df = spark.read.format("binaryFile").option("mimeType", "image/*").load(image_paths)

# Extract the base file name and the type of image
image_df = image_df.withColumn(
    "base_name", 
    F.regexp_replace(F.regexp_extract(F.col("_metadata.file_path"), r'([^/]+)\.[^.]+$', 1), ' ', '_')
)
image_df = image_df.withColumn(
    "directory", 
    F.regexp_replace(F.regexp_extract(F.col("_metadata.file_path"), r'([^/]+)/([^/]+)/[^/]+$', 2), '%20', '_')
)

display(image_df.sort('directory', 'base_name', ascending=False))

#### Display Tissue Cell Images and corresponding Mask Annotations Side-by-side

In [0]:
from pyspark.sql.functions import input_file_name, regexp_extract, col

# Define the project directory and subdirectories to use
directories2use = ['tissue images', 'mask binary without border', 'label masks modify']
valid_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.gif']

# Function to load images from a specific directory within a subdirectory
def load_images_from_directory(subdirectory, directory):
    full_path = f"{subdirectory.path}{directory}/"
    files = dbutils.fs.ls(full_path)
    valid_files = [file.path for file in files if any(file.path.endswith(ext) for ext in valid_extensions)]
    if valid_files:
        image_df = spark.read.format("binaryFile").option("mimeType", "image/*").load(valid_files)
        image_df = image_df.withColumn("base_name", regexp_extract(input_file_name(), r'([^/]+)\.[^.]+$', 1))
        column_name = directory.replace(' ', '_')
        image_df = image_df.withColumnRenamed("content", column_name)
        return image_df.select("base_name", column_name)
    return None

# List to store all DataFrames
common_joined_df = None

# Iterate through the specified subdirectories
for subdirectory in dbutils.fs.ls(PROJECT_RAWDATA_DIR)[2:5]:    
    if subdirectory.isDir() and (subdirectory.name != 'yolo_dataset/'):
        subdirectory_dfs = []
        for directory in directories2use:
            df = load_images_from_directory(subdirectory, directory)
            if df is not None:
                subdirectory_dfs.append(df)
        
        # Join the DataFrames for the current subdirectory on the base_name column
        if len(subdirectory_dfs) > 1:
            joined_subdirectory_df = subdirectory_dfs[0]
            for df in subdirectory_dfs[1:]:
                joined_subdirectory_df = joined_subdirectory_df.join(df, on="base_name", how="outer")
            
            # Append the joined DataFrame to the common DataFrame
            if common_joined_df is None:
                common_joined_df = joined_subdirectory_df
            else:
                common_joined_df = common_joined_df.union(joined_subdirectory_df)

# Display the common joined DataFrame with image thumbnails
if common_joined_df is not None:
    display(common_joined_df)
else:
    print("No valid images found.")